This notebook is supposed to be run on Kaggle Notebooks (as the original dataset is 50GB) with the following dataset as input:
- https://www.kaggle.com/datasets/nih-chest-xrays/data

It will load the dataset and then select `training_size` and `testing_size` images for each class, resize them to `image_size`, and save them to a new directory structure for training and testing. Finally, it will zip the new directory structure for download.

Futhermore, it will also create a CSV file with the image names, corresponding labels, and split information (train/test) and other metadata.

In [ ]:
import os
import pandas as pd
import multiprocessing
from PIL import Image

In [ ]:
data_path = "/kaggle/input/data/"
data_entry_file = "Data_Entry_2017.csv"
data_output_path = "/kaggle/working/nih/data"
train_data_path = os.path.join(data_output_path, "train")
test_data_path = os.path.join(data_output_path, "test")

In [ ]:
classes = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass","Nodule", "Pneumonia", "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "No Finding"]
image_folder_start_index = [1000, 1336000, 3923014, 6585007, 9232004, 11558008, 13774026, 16051010, 18387035, 20945050, 24718000, 28173003]
training_size = 200
testing_size = 50
image_size = 128

In [ ]:
for folder in [train_data_path, test_data_path]:
    os.makedirs(folder, exist_ok=True)
    for cls in classes:
        os.makedirs(os.path.join(folder, cls), exist_ok=True)

In [ ]:
def get_image_folder(image_id):
        image_id = int("".join([c for c in image_id if c.isdigit()]))

        image_folder = 1

        for folder_id, start_index in enumerate(image_folder_start_index):
            if image_id >= start_index:
                image_folder = folder_id + 1

        return image_folder

In [ ]:
# Read the data entry file
# Rows with selected classes for new data entry file [Training + Testing]
# Save the new data entry file
data_entry_path = os.path.join(data_path, data_entry_file)
df = pd.read_csv(data_entry_path)

# Filter rows with classes of interest with each (testing + training) size
df_filtered = pd.DataFrame()
for cls in classes:
    df_cls = df[df['Finding Labels'] ==cls][0:training_size + testing_size]
    
    # Filter the data for training and testing
    df_cls['Training/Testing'] = 'TRAIN'
    df_cls['Training/Testing'] = ['TRAIN'] * training_size + ['TEST'] * testing_size
    
    df_filtered = pd.concat([df_filtered, df_cls])
    print(f"{cls}: {len(df_cls)} rows")

df_filtered = df_filtered.reset_index(drop=True)

# Add a new column for image folder
df_filtered['Image Folder'] = df_filtered['Image Index'].apply(get_image_folder)

# Save the new data entry file
new_data_entry_path = os.path.join(data_output_path, "Data_Entry_2017_filtered.csv")
df_filtered.to_csv(new_data_entry_path, index=False)

In [ ]:
def process_image(row, resize = 512):
    image_path = os.path.join(
        data_path,
        f"images_{row['Image Folder']:03d}/",
        "images/",
        row['Image Index']
    )
    save_path = os.path.join(
        train_data_path if row['Training/Testing'] == 'TRAIN' else test_data_path,
        f"{row['Finding Labels']}/",
        f"{row['Image Index']}"
    )

    if not os.path.exists(save_path):
        # Resize the image if needed
        with Image.open(image_path) as img:
            img = img.resize((resize, resize))
            img.save(save_path)


In [ ]:
# Split the data equally for multiprocessing
num_processes = multiprocessing.cpu_count()
data_chunks = [df_filtered.iloc[i::num_processes] for i in range(num_processes)]

with multiprocessing.Pool(processes=num_processes) as pool:
    for chunk in data_chunks:
        task_args = [(row.to_dict(), image_size) for _, row in chunk.iterrows()]
        pool.starmap(process_image, task_args)

print("Data fetching complete.")
# Total files in each class
for cls in classes:
    train_count = len(os.listdir(os.path.join(train_data_path, cls)))
    test_count = len(os.listdir(os.path.join(test_data_path, cls)))
    print(f"{cls} - TRAIN: {train_count} files, TEST: {test_count} files")


In [ ]:
import shutil
shutil.make_archive(data_output_path, 'zip', "/kaggle/working/nih/")